# Axon exact-v4 Linux read-page diagnostic

Attach `axon_exact_v4_bundle.zip` as a Kaggle dataset, enable GPU, then run.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

work = Path('/kaggle/working/axon_exact_v4')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True, exist_ok=True)

bundle_candidates = (
    sorted(Path('/kaggle/input').glob('**/*.axonbundle'))
    + sorted(Path('/kaggle/input').glob('**/*_bundle.zip'))
)
if not bundle_candidates:
    raise FileNotFoundError('Attach axon_exact_v4_bundle as a Kaggle dataset first.')
bundle = bundle_candidates[0]
print('bundle zip:', bundle)
with zipfile.ZipFile(bundle) as zf:
    zf.extractall(work)

manifest = json.loads((work / 'bundle_manifest.json').read_text())
print(json.dumps(manifest['checkpoint'], indent=2))
print('training:', manifest['training'])

In [ ]:
repo = work / 'axon'
checkpoint = work / manifest['checkpoint']['archive_path']
dataset_rel = Path(manifest['dataset']['archive_path'])
dataset_dir = work / dataset_rel
assert repo.exists(), repo
assert checkpoint.exists(), checkpoint
assert dataset_dir.exists(), dataset_dir

cmd = [
    sys.executable, 'training/diagnose_exact_v4_linux.py',
    str(checkpoint), str(dataset_dir),
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=repo, check=True)